# T31-机构行为监测 - 数据处理模块

## 1. 模块概述

数据处理模块负责对获取的原始数据进行清洗、转换和标准化处理，为后续分析提供高质量的数据基础。

主要功能：
- 数据清洗：处理缺失值、异常值和特殊标记
- 数据转换：类型转换、格式标准化
- 数据聚合：按维度汇总统计
- 期限分类：将剩余期限映射到标准分类

## 2. 数据清洗工具

In [ ]:
import pandas as pd
import numpy as np
import re
import datetime
from typing import Optional, List, Dict, Any, Union

class DataCleaner:
    """数据清洗工具类"""
    
    @staticmethod
    def clean_special_values(df: pd.DataFrame, columns: List[str], 
                            special_value: str = '--', replace_value: Any = 0) -> pd.DataFrame:
        """
        清洗特殊值（如'--'）
        
        Args:
            df: 原始DataFrame
            columns: 需要处理的列名列表
            special_value: 特殊值标记
            replace_value: 替换值
        
        Returns:
            DataFrame: 清洗后的数据
        """
        df = df.copy()
        for col in columns:
            if col in df.columns:
                df[col] = df[col].replace(special_value, replace_value)
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(replace_value)
        return df
    
    @staticmethod
    def clean_date_column(df: pd.DataFrame, column: str = '交易日期', 
                         date_format: str = None) -> pd.DataFrame:
        """
        清洗日期列
        
        Args:
            df: 原始DataFrame
            column: 日期列名
            date_format: 日期格式（可选）
        
        Returns:
            DataFrame: 清洗后的数据
        """
        df = df.copy()
        if column in df.columns:
            df[column] = pd.to_datetime(df[column], format=date_format)
            df[column] = df[column].dt.date
        return df
    
    @staticmethod
    def remove_empty_rows(df: pd.DataFrame, subset: List[str] = None) -> pd.DataFrame:
        """
        移除空行
        
        Args:
            df: 原始DataFrame
            subset: 检查的列名列表
        
        Returns:
            DataFrame: 清理后的数据
        """
        return df.dropna(subset=subset, how='all')
    
    @staticmethod
    def remove_empty_columns(df: pd.DataFrame) -> pd.DataFrame:
        """
        移除空列
        
        Args:
            df: 原始DataFrame
        
        Returns:
            DataFrame: 清理后的数据
        """
        return df.dropna(axis=1, how='all')

# 创建清洗器实例
cleaner = DataCleaner()
print("数据清洗器已创建")

## 3. 期限分类处理

### 3.1 期限映射器

将剩余期限转换为标准分类。

In [ ]:
class TenorMapper:
    """期限分类映射器"""
    
    # 期限分类边界（年）
    TENOR_BOUNDARIES = [
        (0, 1, '1Y以下'),
        (1, 3, '1-3Y'),
        (3, 5, '3-5Y'),
        (5, 7, '5-7Y'),
        (7, 10, '7-10Y'),
        (10, 15, '10-15Y'),
        (15, 20, '15-20Y'),
        (20, 30, '20-30Y'),
        (30, float('inf'), '30Y以上')
    ]
    
    @staticmethod
    def parse_tenor_string(tenor_str: str) -> Optional[float]:
        """
        解析期限字符串为年数
        
        Args:
            tenor_str: 期限字符串（如 '5Y+', '180D'）
        
        Returns:
            float: 年数，解析失败返回None
        """
        if tenor_str is None or pd.isna(tenor_str):
            return None
        
        tenor_str = str(tenor_str).strip()
        
        # 处理 Y+ 格式（如 '5Y+', '10Y+'）
        if 'Y' in tenor_str:
            match = re.match(r'(\d+\.?\d*)Y', tenor_str)
            if match:
                return float(match.group(1))
        
        # 处理 D 格式（如 '180D', '365D'）
        if 'D' in tenor_str:
            match = re.match(r'(\d+\.?\d*)D', tenor_str)
            if match:
                days = float(match.group(1))
                return days / 365
        
        # 尝试直接解析数字
        try:
            return float(tenor_str)
        except ValueError:
            return None
    
    @classmethod
    def map_to_category(cls, years: float) -> str:
        """
        将年数映射到期限分类
        
        Args:
            years: 年数
        
        Returns:
            str: 期限分类
        """
        if years is None or pd.isna(years):
            return '其他'
        
        for min_year, max_year, category in cls.TENOR_BOUNDARIES:
            if min_year < years <= max_year:
                return category
        
        return '其他'
    
    @classmethod
    def categorize_tenor(cls, tenor_series: pd.Series) -> pd.Series:
        """
        批量将期限字符串转换为分类
        
        Args:
            tenor_series: 期限字符串Series
        
        Returns:
            Series: 期限分类Series
        """
        return tenor_series.apply(
            lambda x: cls.map_to_category(cls.parse_tenor_string(x))
        )

# 创建期限映射器实例
tenor_mapper = TenorMapper()
print("期限映射器已创建")

# 测试期限解析
test_tenors = ['1Y+', '3Y+', '5Y+', '10Y+', '180D', '365D']
print("\n期限解析测试:")
for t in test_tenors:
    years = tenor_mapper.parse_tenor_string(t)
    category = tenor_mapper.map_to_category(years)
    print(f"  {t} -> {years}年 -> {category}")

## 4. 交易数据处理

In [ ]:
class TradeDataProcessor:
    """交易数据处理器"""
    
    def __init__(self):
        self.cleaner = DataCleaner()
        self.tenor_mapper = TenorMapper()
    
    def process_institution_stats(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        处理机构统计数据
        
        Args:
            df: 原始机构统计数据
        
        Returns:
            DataFrame: 处理后的数据
        """
        # 清洗交易量列
        volume_columns = ['净买入交易量（亿元）', '买入交易量（亿元）', '卖出交易量（亿元）']
        df = self.cleaner.clean_special_values(df, volume_columns)
        
        # 清洗日期列
        df = self.cleaner.clean_date_column(df, '交易日期')
        
        return df
    
    def calculate_net_inflow(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算净流入量
        
        Args:
            df: 包含买入和卖出量的数据
        
        Returns:
            DataFrame: 包含净流入量的数据
        """
        df = df.copy()
        
        if '买入交易量（亿元）' in df.columns and '卖出交易量（亿元）' in df.columns:
            df['净流入'] = df['买入交易量（亿元）'] - df['卖出交易量（亿元）']
        
        return df
    
    def calculate_total_volume(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算总交易量
        
        Args:
            df: 包含买入和卖出量的数据
        
        Returns:
            DataFrame: 包含总交易量的数据
        """
        df = df.copy()
        
        if '买入交易量（亿元）' in df.columns and '卖出交易量（亿元）' in df.columns:
            df['总交易量'] = df['买入交易量（亿元）'] + df['卖出交易量（亿元）']
        elif 'buy_vol' in df.columns and 'sell_vol' in df.columns:
            df['总交易量'] = df['buy_vol'] + df['sell_vol']
        
        return df
    
    def calculate_percentage(self, df: pd.DataFrame, value_col: str, 
                            result_col: str = 'percentage') -> pd.DataFrame:
        """
        计算百分比
        
        Args:
            df: 数据
            value_col: 值列名
            result_col: 结果列名
        
        Returns:
            DataFrame: 包含百分比的数据
        """
        df = df.copy()
        total = df[value_col].sum()
        
        if total > 0:
            df[result_col] = (df[value_col] / total) * 100
        else:
            df[result_col] = 0
        
        return df

# 创建交易数据处理器
trade_processor = TradeDataProcessor()
print("交易数据处理器已创建")

## 5. 市场份额分析处理

In [ ]:
class MarketShareProcessor:
    """市场份额分析处理器"""
    
    def __init__(self):
        self.cleaner = DataCleaner()
    
    def calculate_daily_market_share(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        计算每日市场份额
        
        Args:
            df: 包含交易日期、机构类型和交易量的数据
        
        Returns:
            DataFrame: 市场份额数据
        """
        df = df.copy()
        
        # 计算总交易量
        if '总交易量' not in df.columns:
            df = self.cleaner.clean_special_values(df, ['买入交易量（亿元）', '卖出交易量（亿元）'])
            df['总交易量'] = df['买入交易量（亿元）'] + df['卖出交易量（亿元）']
        
        # 计算每日市场总量
        daily_total = df.groupby('交易日期')['总交易量'].sum().reset_index()
        daily_total.rename(columns={'总交易量': '市场总量'}, inplace=True)
        
        # 计算每日机构交易量
        daily_inst = df.groupby(['交易日期', '机构类型'])['总交易量'].sum().reset_index()
        
        # 合并计算市场份额
        result = pd.merge(daily_inst, daily_total, on='交易日期')
        result['市场份额(%)'] = (result['总交易量'] / result['市场总量']) * 100
        
        return result
    
    def pivot_market_share(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        将市场份额数据透视为宽表格式
        
        Args:
            df: 市场份额数据（长表）
        
        Returns:
            DataFrame: 宽表格式的市场份额
        """
        return df.pivot(
            index='交易日期', 
            columns='机构类型', 
            values='市场份额(%)'
        ).fillna(0)

# 创建市场份额处理器
market_share_processor = MarketShareProcessor()
print("市场份额处理器已创建")

## 6. 利差计算处理

In [ ]:
class SpreadCalculator:
    """利差计算器"""
    
    @staticmethod
    def calculate_term_spread(df_10y: pd.DataFrame, df_1y: pd.DataFrame) -> pd.DataFrame:
        """
        计算期限利差（10Y - 1Y）
        
        Args:
            df_10y: 10年期国债收益率数据
            df_1y: 1年期国债收益率数据
        
        Returns:
            DataFrame: 期限利差数据
        """
        df_merged = pd.merge(df_10y, df_1y, on='DT', suffixes=('_10y', '_1y'))
        df_merged['term_spread'] = df_merged['yield_10y'] - df_merged['yield_1y']
        
        return df_merged[['DT', 'term_spread']]
    
    @staticmethod
    def calculate_credit_spread(df_credit: pd.DataFrame, df_gov: pd.DataFrame) -> pd.DataFrame:
        """
        计算信用利差
        
        Args:
            df_credit: 信用债收益率数据
            df_gov: 国债收益率数据
        
        Returns:
            DataFrame: 信用利差数据
        """
        df_merged = pd.merge(df_credit, df_gov, on='DT', suffixes=('_credit', '_gov'))
        df_merged['credit_spread'] = (df_merged['yield_credit'] - df_merged['yield_gov']) * 100  # BP
        
        return df_merged[['DT', 'credit_spread']]
    
    @staticmethod
    def calculate_city_bond_spread(df_city: pd.DataFrame, df_gov: pd.DataFrame) -> pd.DataFrame:
        """
        计算城投债利差
        
        Args:
            df_city: 城投债收益率数据
            df_gov: 国债收益率数据（按期限）
        
        Returns:
            DataFrame: 城投债利差数据
        """
        df_merged = pd.merge(df_city, df_gov, on='tenor', how='left')
        df_merged.dropna(subset=['weighted_YTM', 'risk_free_rate'], inplace=True)
        df_merged['spread'] = (df_merged['weighted_YTM'] - df_merged['risk_free_rate']) * 100  # BP
        
        return df_merged

# 创建利差计算器
spread_calculator = SpreadCalculator()
print("利差计算器已创建")

## 7. 资产分类处理

In [ ]:
class AssetClassifier:
    """资产分类器"""
    
    # 债券类型到资产大类的映射
    BOND_TYPE_MAPPING = {
        # 利率债
        '国债': '利率债',
        '地方政府债': '利率债',
        '央行票据': '利率债',
        # 金融债
        '金融债': '金融债',
        '同业存单': '金融债',
        '政策性金融债': '金融债',
        # 信用债
        '中期票据': '信用债',
        '企业债': '信用债',
        '公司债': '信用债',
        '短期融资券': '信用债',
        '超短期融资券': '信用债',
        '定向工具': '信用债',
        'ABS': '信用债',
        '可转债': '信用债',
    }
    
    @classmethod
    def classify_bond(cls, bond_type: str) -> str:
        """
        将债券类型映射到资产大类
        
        Args:
            bond_type: 债券类型
        
        Returns:
            str: 资产大类
        """
        return cls.BOND_TYPE_MAPPING.get(bond_type, '信用债')
    
    @classmethod
    def add_asset_class_column(cls, df: pd.DataFrame, bond_type_col: str = '债券类型') -> pd.DataFrame:
        """
        添加资产大类列
        
        Args:
            df: 原始数据
            bond_type_col: 债券类型列名
        
        Returns:
            DataFrame: 添加了资产大类的数据
        """
        df = df.copy()
        if bond_type_col in df.columns:
            df['资产大类'] = df[bond_type_col].apply(cls.classify_bond)
        return df

# 创建资产分类器
asset_classifier = AssetClassifier()
print("资产分类器已创建")

# 测试资产分类
test_bond_types = ['国债', '金融债', '中期票据', '企业债', 'ABS']
print("\n资产分类测试:")
for bt in test_bond_types:
    asset_class = asset_classifier.classify_bond(bt)
    print(f"  {bt} -> {asset_class}")

## 8. 省份名称清理

In [ ]:
class ProvinceNameCleaner:
    """省份名称清理器"""
    
    # 需要移除的后缀
    SUFFIXES_TO_REMOVE = [
        '省', '市', '自治区', 
        '回族自治区', '维吾尔自治区', '壮族自治区',
        '特别行政区'
    ]
    
    @classmethod
    def clean_province_name(cls, name: str) -> str:
        """
        清理省份名称
        
        Args:
            name: 原始省份名称
        
        Returns:
            str: 清理后的省份名称
        """
        if pd.isna(name):
            return name
        
        # 按长度降序排序，确保先匹配长的后缀
        sorted_suffixes = sorted(cls.SUFFIXES_TO_REMOVE, key=len, reverse=True)
        
        for suffix in sorted_suffixes:
            if suffix in name:
                name = name.replace(suffix, '')
        
        return name
    
    @classmethod
    def clean_province_series(cls, series: pd.Series) -> pd.Series:
        """
        批量清理省份名称
        
        Args:
            series: 省份名称Series
        
        Returns:
            Series: 清理后的省份名称
        """
        return series.apply(cls.clean_province_name)

# 创建省份名称清理器
province_cleaner = ProvinceNameCleaner()
print("省份名称清理器已创建")

# 测试省份名称清理
test_provinces = ['江苏省', '上海市', '新疆维吾尔自治区', '广西壮族自治区', '宁夏回族自治区']
print("\n省份名称清理测试:")
for p in test_provinces:
    cleaned = province_cleaner.clean_province_name(p)
    print(f"  {p} -> {cleaned}")

## 9. 数据聚合工具

In [ ]:
class DataAggregator:
    """数据聚合工具"""
    
    @staticmethod
    def aggregate_by_institution(df: pd.DataFrame, value_cols: List[str],
                                 agg_func: str = 'sum') -> pd.DataFrame:
        """
        按机构聚合
        
        Args:
            df: 原始数据
            value_cols: 值列名列表
            agg_func: 聚合函数
        
        Returns:
            DataFrame: 聚合结果
        """
        return df.groupby('机构类型')[value_cols].agg(agg_func).reset_index()
    
    @staticmethod
    def aggregate_by_date(df: pd.DataFrame, value_cols: List[str],
                         agg_func: str = 'sum') -> pd.DataFrame:
        """
        按日期聚合
        
        Args:
            df: 原始数据
            value_cols: 值列名列表
            agg_func: 聚合函数
        
        Returns:
            DataFrame: 聚合结果
        """
        return df.groupby('交易日期')[value_cols].agg(agg_func).reset_index()
    
    @staticmethod
    def aggregate_by_tenor(df: pd.DataFrame, value_cols: List[str],
                          agg_func: str = 'sum') -> pd.DataFrame:
        """
        按期限聚合
        
        Args:
            df: 原始数据
            value_cols: 值列名列表
            agg_func: 聚合函数
        
        Returns:
            DataFrame: 聚合结果
        """
        return df.groupby('期限')[value_cols].agg(agg_func).reset_index()
    
    @staticmethod
    def aggregate_by_bond_type(df: pd.DataFrame, value_cols: List[str],
                               agg_func: str = 'sum') -> pd.DataFrame:
        """
        按债券类型聚合
        
        Args:
            df: 原始数据
            value_cols: 值列名列表
            agg_func: 聚合函数
        
        Returns:
            DataFrame: 聚合结果
        """
        return df.groupby('债券类型')[value_cols].agg(agg_func).reset_index()

# 创建聚合器
aggregator = DataAggregator()
print("数据聚合器已创建")

## 10. 统一数据处理流水线

In [ ]:
class DataProcessingPipeline:
    """统一数据处理流水线"""
    
    def __init__(self):
        """初始化流水线"""
        self.cleaner = DataCleaner()
        self.tenor_mapper = TenorMapper()
        self.trade_processor = TradeDataProcessor()
        self.market_share_processor = MarketShareProcessor()
        self.spread_calculator = SpreadCalculator()
        self.asset_classifier = AssetClassifier()
        self.province_cleaner = ProvinceNameCleaner()
        self.aggregator = DataAggregator()
    
    def process_for_chart1(self, df: pd.DataFrame, trade_date: str) -> pd.DataFrame:
        """
        处理图表1数据：各机构类型每日净交易量
        """
        # 清洗数据
        df = self.cleaner.clean_special_values(df, ['净买入交易量（亿元）'])
        
        # 按机构类型聚合
        df_result = df.groupby('机构类型')['净买入交易量（亿元）'].sum().reset_index()
        df_result.rename(columns={'机构类型': 'name', '净买入交易量（亿元）': 'value'}, inplace=True)
        
        return df_result
    
    def process_for_chart3(self, df: pd.DataFrame, trade_date: str, institution_type: str) -> pd.DataFrame:
        """
        处理图表3数据：机构期限偏好分析
        """
        # 筛选机构
        df = df[df['机构类型'] == institution_type].copy()
        
        # 清洗数据
        df = self.cleaner.clean_special_values(df, ['买入交易量（亿元）', '卖出交易量（亿元）'])
        
        # 计算总交易量
        df['total_vol'] = df['买入交易量（亿元）'] + df['卖出交易量（亿元）']
        df = df[df['total_vol'] > 0]
        
        # 计算百分比
        total = df['total_vol'].sum()
        df['percent'] = (df['total_vol'] / total * 100) if total > 0 else 0
        
        df.rename(columns={'期限': 'name', 'total_vol': 'value'}, inplace=True)
        
        return df[['name', 'value', 'percent']]
    
    def process_for_chart5(self, df: pd.DataFrame, start_date: str, end_date: str) -> pd.DataFrame:
        """
        处理图表5数据：市场份额时序分析
        """
        # 计算市场份额
        df_market_share = self.market_share_processor.calculate_daily_market_share(df)
        
        # 透视为宽表
        df_pivot = self.market_share_processor.pivot_market_share(df_market_share)
        
        return df_pivot
    
    def process_for_chart12(self, df: pd.DataFrame, start_date: str, end_date: str, 
                            institution_type: str) -> pd.DataFrame:
        """
        处理图表12数据：机构大类券种配置流向
        """
        # 筛选机构和日期
        df = df[(df['机构类型'] == institution_type)].copy()
        
        # 清洗数据
        df = self.cleaner.clean_special_values(df, ['买入交易量（亿元）', '卖出交易量（亿元）'])
        
        # 计算净流入
        df['net_flow'] = df['买入交易量（亿元）'] - df['卖出交易量（亿元）']
        
        # 添加资产大类
        df = self.asset_classifier.add_asset_class_column(df)
        
        # 按资产大类聚合
        df_result = df.groupby('资产大类')['net_flow'].sum().reset_index()
        df_result.rename(columns={'资产大类': 'name', 'net_flow': 'value'}, inplace=True)
        
        return df_result

# 创建数据处理流水线
pipeline = DataProcessingPipeline()
print("数据处理流水线已创建")

In [ ]:
# 模块总结
print("=" * 60)
print("数据处理模块组件列表")
print("=" * 60)
print("\n1. DataCleaner - 数据清洗工具")
print("   - clean_special_values(): 清洗特殊值")
print("   - clean_date_column(): 清洗日期列")
print("   - remove_empty_rows(): 移除空行")
print("   - remove_empty_columns(): 移除空列")
print("\n2. TenorMapper - 期限分类映射器")
print("   - parse_tenor_string(): 解析期限字符串")
print("   - map_to_category(): 映射到分类")
print("   - categorize_tenor(): 批量转换")
print("\n3. TradeDataProcessor - 交易数据处理器")
print("   - process_institution_stats(): 处理机构统计")
print("   - calculate_net_inflow(): 计算净流入")
print("   - calculate_total_volume(): 计算总交易量")
print("   - calculate_percentage(): 计算百分比")
print("\n4. MarketShareProcessor - 市场份额处理器")
print("   - calculate_daily_market_share(): 计算每日市场份额")
print("   - pivot_market_share(): 透视为宽表")
print("\n5. SpreadCalculator - 利差计算器")
print("   - calculate_term_spread(): 计算期限利差")
print("   - calculate_credit_spread(): 计算信用利差")
print("   - calculate_city_bond_spread(): 计算城投债利差")
print("\n6. AssetClassifier - 资产分类器")
print("   - classify_bond(): 债券分类")
print("   - add_asset_class_column(): 添加资产大类列")
print("\n7. ProvinceNameCleaner - 省份名称清理器")
print("   - clean_province_name(): 清理省份名称")
print("   - clean_province_series(): 批量清理")
print("\n8. DataAggregator - 数据聚合工具")
print("   - aggregate_by_institution(): 按机构聚合")
print("   - aggregate_by_date(): 按日期聚合")
print("   - aggregate_by_tenor(): 按期限聚合")
print("   - aggregate_by_bond_type(): 按债券类型聚合")
print("\n9. DataProcessingPipeline - 统一处理流水线")
print("   - process_for_chart1(): 处理图表1数据")
print("   - process_for_chart3(): 处理图表3数据")
print("   - process_for_chart5(): 处理图表5数据")
print("   - process_for_chart12(): 处理图表12数据")
print("\n" + "=" * 60)
print("数据处理模块加载完成")
print("=" * 60)